<a href="https://colab.research.google.com/github/DevD231204/Quantum-Mini-Project/blob/main/QNBC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip -q install qiskit qiskit-aer

In [4]:
import math
import numpy as np
import pandas as pd

from itertools import product
from sklearn.model_selection import train_test_split

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import RYGate
from qiskit_aer import AerSimulator

from IPython.display import display

Load the Titanic dataset

In [5]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(url)

df = df[
    ["Survived", "Sex", "Age", "Pclass", "SibSp"]
].copy()

print("Dataset shape:", df.shape)
print("\nFirst five records:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (891, 5)

First five records:


,Survived,Sex,Age,Pclass,SibSp
0,0,male,22.0,3,1
1,1,female,38.0,1,1
2,1,female,26.0,3,0
3,1,female,35.0,1,1
4,0,male,35.0,3,0



Missing values:
Survived      0
Sex           0
Age         177
Pclass        0
SibSp         0
dtype: int64


Preprocessing and binary encoding

In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Survived"]
)

age_median = train_df["Age"].median()

feature_names = [
    "Female",
    "Child",
    "FirstClass",
    "Family"
]

def preprocess(data):
    age = data["Age"].fillna(age_median)

    X = pd.DataFrame({
        "Female": (
            data["Sex"].str.lower() == "female"
        ).astype(int),

        "Child": (
            age < 16
        ).astype(int),

        "FirstClass": (
            data["Pclass"] == 1
        ).astype(int),

        "Family": (
            data["SibSp"] > 0
        ).astype(int)
    }, index=data.index)

    return X

X_train = preprocess(train_df).to_numpy(dtype=int)
X_test = preprocess(test_df).to_numpy(dtype=int)

y_train = train_df["Survived"].to_numpy(dtype=int)
y_test = test_df["Survived"].to_numpy(dtype=int)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nFeature example:")
print(X_train[:5])

print("\nCorresponding labels:")
print(y_train[:5])

Training samples: 712
Testing samples: 179

Feature example:
[[0 0 0 0]
 [0 0 0 0]
 [0 0 1 0]
 [1 0 0 0]
 [1 0 0 1]]

Corresponding labels:
[1 0 0 1 1]


Estimate the Naive Bayes probability parameters

In [7]:
ALPHA = 1.0

classes = [0, 1]

N = len(y_train)

class_counts = {
    c: int(np.sum(y_train == c))
    for c in classes
}

priors = {
    c: class_counts[c] / N
    for c in classes
}

conditional_probs = {}

for c in classes:

    class_samples = X_train[y_train == c]

    conditional_probs[c] = []

    for j in range(4):

        count_ones = int(np.sum(class_samples[:, j]))

        probability = (
            count_ones + ALPHA
        ) / (
            class_counts[c] + 2 * ALPHA
        )

        conditional_probs[c].append(probability)

print("CLASS PRIORS")
for c in classes:
    print(f"P(C={c}) = {priors[c]:.6f}")

print("\nCONDITIONAL PROBABILITIES")

for c in classes:
    print(f"\nClass {c}:")

    for feature, probability in zip(
        feature_names,
        conditional_probs[c]
    ):
        print(
            f"P({feature}=1 | C={c}) = {probability:.6f}"
        )

CLASS PRIORS
P(C=0) = 0.616573
P(C=1) = 0.383427

CONDITIONAL PROBABILITIES

Class 0:
P(Female=1 | C=0) = 0.149660
P(Child=1 | C=0) = 0.063492
P(FirstClass=1 | C=0) = 0.138322
P(Family=1 | C=0) = 0.263039

Class 1:
P(Female=1 | C=1) = 0.687273
P(Child=1 | C=1) = 0.145455
P(FirstClass=1 | C=1) = 0.407273
P(Family=1 | C=1) = 0.378182


Constructing the five-qubit quantum circuit

In [8]:
# Create a 5-qubit, 5-classical-bit circuit
qc = QuantumCircuit(5, 5)

# ------------------------------------------------
# STEP A: Encode class prior on q[0]
# ------------------------------------------------

p_survived = priors[1]

theta_class = 2 * math.asin(
    math.sqrt(p_survived)
)

qc.ry(theta_class, 0)

# ------------------------------------------------
# STEP B: Encode conditional probabilities
# ------------------------------------------------

for i in range(4):

    p_class_0 = conditional_probs[0][i]
    p_class_1 = conditional_probs[1][i]

    angle_0 = 2 * math.asin(
        math.sqrt(p_class_0)
    )

    angle_1 = 2 * math.asin(
        math.sqrt(p_class_1)
    )

    # Rotation when class qubit is |0>
    gate_0 = RYGate(angle_0).control(
        num_ctrl_qubits=1,
        ctrl_state=0
    )

    # Rotation when class qubit is |1>
    gate_1 = RYGate(angle_1).control(
        num_ctrl_qubits=1,
        ctrl_state=1
    )

    qc.append(gate_0, [0, i + 1])
    qc.append(gate_1, [0, i + 1])

# ------------------------------------------------
# STEP C: Measurement
# ------------------------------------------------

qc.measure(range(5), range(5))

print(qc.draw(output="text"))

     ┌────────────┐                                               »
q_0: ┤ Ry(1.3355) ├───────o─────────────■──────────────o──────────»
     └────────────┘┌──────┴──────┐┌─────┴──────┐       │       ┌─┐»
q_1: ──────────────┤ Ry(0.79445) ├┤ Ry(1.9547) ├───────┼───────┤M├»
                   └─────────────┘└────────────┘┌──────┴──────┐└╥┘»
q_2: ───────────────────────────────────────────┤ Ry(0.50944) ├─╫─»
                                                └─────────────┘ ║ »
q_3: ───────────────────────────────────────────────────────────╫─»
                                                                ║ »
q_4: ───────────────────────────────────────────────────────────╫─»
                                                                ║ »
c: 5/═══════════════════════════════════════════════════════════╩═»
                                                                1 »
«                                                                     »
«q_0: ───────■──────────────o───────────────

In [9]:
SHOTS = 50000

simulator = AerSimulator()

compiled_circuit = transpile(
    qc,
    simulator,
    optimization_level=1
)

result = simulator.run(
    compiled_circuit,
    shots=SHOTS,
    seed_simulator=42
).result()

counts = result.get_counts()

print("TOTAL SHOTS:", SHOTS)

print("\nMEASUREMENT COUNTS:")
print(counts)

TOTAL SHOTS: 50000

MEASUREMENT COUNTS:
{'11110': 6, '01110': 29, '11010': 160, '01000': 2451, '01100': 171, '01001': 1305, '10001': 1171, '00101': 314, '00010': 2739, '01011': 2869, '11101': 147, '00011': 4171, '00111': 736, '01111': 495, '00100': 1071, '10000': 5571, '00110': 193, '00001': 1885, '10100': 382, '10111': 428, '11100': 73, '10010': 996, '10101': 198, '01101': 228, '10011': 2473, '11011': 1631, '11111': 291, '11000': 946, '10110': 68, '00000': 15614, '11001': 740, '01010': 448}


Decode the measurement results

In [10]:
# Count measurements for each class
quantum_class_counts = np.zeros(2, dtype=int)

# Count feature value 1 for each class
quantum_feature_ones = np.zeros((2, 4), dtype=int)

# Count every possible joint outcome
joint_counts = {
    (c, x): 0
    for c in classes
    for x in product([0, 1], repeat=4)
}

for bitstring, frequency in counts.items():

    # Convert binary measurement string to integer
    value = int(bitstring.replace(" ", ""), 2)

    # q[0] is the least significant bit
    predicted_class = value & 1

    # Extract the four feature bits
    features = tuple(
        (value >> (i + 1)) & 1
        for i in range(4)
    )

    quantum_class_counts[predicted_class] += frequency

    for i, feature_value in enumerate(features):
        if feature_value == 1:
            quantum_feature_ones[predicted_class, i] += frequency

    joint_counts[(predicted_class, features)] += frequency


# ------------------------------------------------
# Estimate quantum class priors
# ------------------------------------------------

quantum_priors = {
    c: quantum_class_counts[c] / SHOTS
    for c in classes
}

# ------------------------------------------------
# Estimate quantum conditional probabilities
# ------------------------------------------------

quantum_conditional_probs = {}

for c in classes:

    if quantum_class_counts[c] == 0:
        raise ValueError(
            f"No measurements obtained for class {c}"
        )

    quantum_conditional_probs[c] = [
        quantum_feature_ones[c, i] / quantum_class_counts[c]
        for i in range(4)
    ]


print("QUANTUM-ESTIMATED CLASS PRIORS")

for c in classes:
    print(
        f"P(C={c}) = {quantum_priors[c]:.6f}"
    )

print("\nQUANTUM-ESTIMATED CONDITIONAL PROBABILITIES")

for c in classes:

    print(f"\nClass {c}:")

    for feature, probability in zip(
        feature_names,
        quantum_conditional_probs[c]
    ):

        print(
            f"P({feature}=1 | C={c}) = {probability:.6f}"
        )

QUANTUM-ESTIMATED CLASS PRIORS
P(C=0) = 0.618360
P(C=1) = 0.381640

QUANTUM-ESTIMATED CONDITIONAL PROBABILITIES

Class 0:
P(Female=1 | C=0) = 0.150042
P(Child=1 | C=0) = 0.064461
P(FirstClass=1 | C=0) = 0.138560
P(Family=1 | C=0) = 0.265282

Class 1:
P(Female=1 | C=1) = 0.686196
P(Child=1 | C=1) = 0.148674
P(FirstClass=1 | C=1) = 0.403836
P(Family=1 | C=1) = 0.370978


Comparison

In [11]:
# ------------------------------------------------
# Compare class priors
# ------------------------------------------------

print("CLASS PRIOR COMPARISON")

for c in classes:

    error = abs(
        priors[c] - quantum_priors[c]
    )

    print(
        f"Class {c}: Classical={priors[c]:.6f}, "
        f"Quantum={quantum_priors[c]:.6f}, "
        f"Error={error:.6f}"
    )


# ------------------------------------------------
# Compare conditional probabilities
# ------------------------------------------------

comparison_rows = []

for c in classes:

    for i, feature in enumerate(feature_names):

        classical_value = conditional_probs[c][i]
        quantum_value = quantum_conditional_probs[c][i]

        comparison_rows.append({
            "Class": c,
            "Feature": feature,
            "Classical": classical_value,
            "Quantum": quantum_value,
            "Absolute Error": abs(
                classical_value - quantum_value
            )
        })

comparison_df = pd.DataFrame(comparison_rows)

print("\nCONDITIONAL PROBABILITY COMPARISON")
display(comparison_df)


# ------------------------------------------------
# Calculate theoretical joint probabilities
# ------------------------------------------------

theoretical_joint = {}

for c in classes:

    for x in product([0, 1], repeat=4):

        probability = priors[c]

        for i, value in enumerate(x):

            p = conditional_probs[c][i]

            probability *= p if value == 1 else (1 - p)

        theoretical_joint[(c, x)] = probability


# ------------------------------------------------
# Calculate empirical joint probabilities
# ------------------------------------------------

empirical_joint = {
    state: joint_counts[state] / SHOTS
    for state in joint_counts
}


# ------------------------------------------------
# Total variation distance
# ------------------------------------------------

tv_distance = 0.5 * sum(
    abs(
        theoretical_joint[state] -
        empirical_joint[state]
    )
    for state in theoretical_joint
)

print("\nTOTAL VARIATION DISTANCE:")
print(f"{tv_distance:.6f}")

CLASS PRIOR COMPARISON
Class 0: Classical=0.616573, Quantum=0.618360, Error=0.001787
Class 1: Classical=0.383427, Quantum=0.381640, Error=0.001787

CONDITIONAL PROBABILITY COMPARISON


,Class,Feature,Classical,Quantum,Absolute Error
0,0,Female,0.149660,0.150042,0.000382
1,0,Child,0.063492,0.064461,0.000969
2,0,FirstClass,0.138322,0.138560,0.000238
3,0,Family,0.263039,0.265282,0.002244
4,1,Female,0.687273,0.686196,0.001076
5,1,Child,0.145455,0.148674,0.003220
6,1,FirstClass,0.407273,0.403836,0.003437
7,1,Family,0.378182,0.370978,0.007204



TOTAL VARIATION DISTANCE:
0.005635


Joint Probability

In [12]:
joint_comparison = []

for state in theoretical_joint:

    c, x = state

    joint_comparison.append({
        "Class": c,
        "Features": x,
        "Theoretical Probability":
            theoretical_joint[state],
        "Measured Probability":
            empirical_joint[state]
    })

joint_df = pd.DataFrame(joint_comparison)

joint_df = joint_df.sort_values(
    by="Theoretical Probability",
    ascending=False
)

print("TOP 10 JOINT PROBABILITY OUTCOMES")

display(joint_df.head(10))

TOP 10 JOINT PROBABILITY OUTCOMES


,Class,Features,Theoretical Probability,Measured Probability
0,0,"(0, 0, 0, 0)",0.311802,0.31228
1,0,"(0, 0, 0, 1)",0.111289,0.11142
24,1,"(1, 0, 0, 0)",0.082998,0.08342
26,1,"(1, 0, 1, 0)",0.057029,0.05738
8,0,"(1, 0, 0, 0)",0.054877,0.05478
25,1,"(1, 0, 0, 1)",0.050478,0.04946
2,0,"(0, 0, 1, 0)",0.050052,0.04902
16,1,"(0, 0, 0, 0)",0.037766,0.03770
27,1,"(1, 0, 1, 1)",0.034684,0.03262
18,1,"(0, 0, 1, 0)",0.025950,0.02610


Prediction Model

In [13]:
def predict_posterior(x, model_priors, model_conditional):

    log_scores = {}

    for c in classes:

        score = math.log(model_priors[c])

        for i, value in enumerate(x):

            p = model_conditional[c][i]

            if value == 1:
                score += math.log(p)
            else:
                score += math.log(1 - p)

        log_scores[c] = score

    # Stable posterior normalization
    maximum = max(log_scores.values())

    exp_scores = {
        c: math.exp(log_scores[c] - maximum)
        for c in classes
    }

    total = sum(exp_scores.values())

    posterior = {
        c: exp_scores[c] / total
        for c in classes
    }

    predicted_class = max(
        posterior,
        key=posterior.get
    )

    return predicted_class, posterior


# ------------------------------------------------
# Test the first passenger
# ------------------------------------------------

sample = X_test[0]

classical_prediction, classical_posterior = predict_posterior(
    sample,
    priors,
    conditional_probs
)

quantum_prediction, quantum_posterior = predict_posterior(
    sample,
    quantum_priors,
    quantum_conditional_probs
)

print("PASSENGER FEATURES:", sample)

print("\nCLASSICAL MODEL")
print("Prediction:", classical_prediction)
print("Posterior:", classical_posterior)

print("\nQUANTUM-ESTIMATED MODEL")
print("Prediction:", quantum_prediction)
print("Posterior:", quantum_posterior)

print("\nACTUAL SURVIVAL LABEL:", y_test[0])

PASSENGER FEATURES: [0 0 0 1]

CLASSICAL MODEL
Prediction: 0
Posterior: {0: 0.8289201834858781, 1: 0.1710798165141219}

QUANTUM-ESTIMATED MODEL
Prediction: 0
Posterior: {0: 0.8328668108404558, 1: 0.16713318915954425}

ACTUAL SURVIVAL LABEL: 0


Compare Accuracy

In [14]:
classical_predictions = []
quantum_predictions = []

for passenger in X_test:

    pred_classical, _ = predict_posterior(
        passenger,
        priors,
        conditional_probs
    )

    pred_quantum, _ = predict_posterior(
        passenger,
        quantum_priors,
        quantum_conditional_probs
    )

    classical_predictions.append(pred_classical)
    quantum_predictions.append(pred_quantum)


classical_accuracy = np.mean(
    np.array(classical_predictions) == y_test
)

quantum_accuracy = np.mean(
    np.array(quantum_predictions) == y_test
)

print("CLASSICAL NAIVE BAYES ACCURACY:")
print(f"{classical_accuracy * 100:.2f}%")

print("\nQUANTUM-ESTIMATED NAIVE BAYES ACCURACY:")
print(f"{quantum_accuracy * 100:.2f}%")

print("\nACCURACY DIFFERENCE:")
print(
    f"{abs(classical_accuracy - quantum_accuracy) * 100:.2f} percentage points"
)

CLASSICAL NAIVE BAYES ACCURACY:
78.21%

QUANTUM-ESTIMATED NAIVE BAYES ACCURACY:
78.21%

ACCURACY DIFFERENCE:
0.00 percentage points
